In [ ]:
from datetime import datetime
from getpass import getpass

# 本ノートブックは テスト手順-管理者機能-S3-機関ストレージ.ipynb の複製に
# SigV4 回帰(S-0〜S-5)のセルを足したもの。元のノートブックは変更していない。

admin_rdm_url = 'https://admin.test.rdm.example.com/'
rdm_url = 'https://test.rdm.example.com/'
idp_name_1 = 'GakuNin RDM IdP'


idp_username_1 = None
idp_password_1 = None

# admin_idp_name を指定すると Embedded DS(IdP選択) 経由でログインする
admin_idp_name = None
admin_username = None
admin_password = None
# entityID 直接指定（Embedded DS バイパス用。管理者ログインがメール/パスワード形式でない環境でのみ必要）
admin_idp_entity_id = None
default_result_path = None
close_on_fail = False
transition_timeout = 60000
skip_failed_test = True
exclude_notebooks = []

# 機関ストレージ設定用パラメータ
target_organization = None

# Amazon S3 設定
# 資格情報は .config.yaml か実行時プロンプトで与える。**ここに値を書かない**。
s3_access_key = None
s3_secret_key = None

# S-1 / S-3 / S-5 用。通常バケット。s3_region_default に作ったもの
s3_bucket = None
# S-4 用。バージョニングを有効にしたバケット。s3_region_default に作ったもの
s3_bucket_versioned = None
# S-2 用。s3_region_explicit に作ったもの
s3_bucket_apne1 = None

# リージョンは管理画面でもアドオン画面でも入力しない(セル [5] の「リージョンの
# 決まり方」参照)。この 2 つは boto3 のクライアント生成と証跡の記録にだけ使う。
s3_region_default = 'us-east-1'
s3_region_explicit = 'ap-northeast-1'

# Server Side Encryption: True = Yes
s3_server_side_encryption = False

# 機関ストレージの表示名（ファイルツリーに表示される名前）
institutional_storage_name = 'NII Storage'

# S3 アドオン(外部ストレージ)の表示名と ID。S-3 / S-4 / S-5 で使う
addon_storage_name = 'Amazon S3'
addon_storage_id = 's3'

# シナリオの実行スイッチ
run_apne1 = True
run_toomany = True
run_versioned = True
run_intra = True
# S-4 の X9 実機版(同一 Key に 1001 版)。既定は無効
run_versioned_bulk = False

toomany_count = 1500
toomany_prefix = 'toomany/'
intra_copy_dest = 'intra-copy-dest/'
intra_move_dest = 'intra-move-dest/'
intra_file_size = 6 * 1024 * 1024

# 証跡の保存先。既定は ../aws-s3-sigv4-improvement/evidence/e2e/run-<YYYYMMDD-HHMMSS>
evidence_dir = None

# プロジェクト名プレフィックス
rdm_project_prefix = 'TEST-S3-SIGV4-{}'.format(datetime.now().strftime('%Y%m%d-%H%M%S'))

In [ ]:
# .config.yaml があれば、未設定(None)のパラメータを同名キーで補完する
# （papermill実行時は先にパラメータが注入されるため、この処理は上書きしない）
import os as _os
import yaml as _yaml

_cfg_path = '.config.yaml'
if _os.path.exists(_cfg_path):
    with open(_cfg_path) as _f:
        _cfg = _yaml.safe_load(_f) or {}
    _loaded = []
    for _k, _v in _cfg.items():
        if _k in globals() and globals()[_k] is None and _v is not None:
            globals()[_k] = _v
            _loaded.append(_k)
    print(f'.config.yaml から補完: {_loaded}')
else:
    print('.config.yaml なし（プロンプト入力で続行）')

In [ ]:
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
if admin_username is None:
    admin_username = input(prompt='Admin Email (管理者画面ログイン用)')
if admin_password is None:
    admin_password = getpass(prompt=f'Password for {admin_username} (管理者画面)')
(len(idp_username_1), len(idp_password_1), len(admin_username), len(admin_password))

In [ ]:
if s3_access_key is None:
    s3_access_key = input(prompt='S3 Access Key')
if s3_secret_key is None:
    s3_secret_key = getpass(prompt='S3 Secret Key')
if s3_bucket is None:
    s3_bucket = input(prompt=f'S3 Bucket Name (通常 / {s3_region_default})')
if run_apne1 and s3_bucket_apne1 is None:
    s3_bucket_apne1 = input(prompt=f'S3 Bucket Name (明示リージョン / {s3_region_explicit})')
if run_versioned and s3_bucket_versioned is None:
    s3_bucket_versioned = input(prompt=f'S3 Bucket Name (バージョニング有効 / {s3_region_default})')

# バケット名は証跡には残すが、ここでは長さだけ確認する(取り違え防止)
assert len({b for b in [s3_bucket, s3_bucket_apne1, s3_bucket_versioned] if b}) == \
    len([b for b in [s3_bucket, s3_bucket_apne1, s3_bucket_versioned] if b]), \
    '3 つのバケットは別名でなければならない'
(len(s3_access_key), len(s3_secret_key), len(s3_bucket))

In [ ]:
import json
import os
import tempfile

work_dir = tempfile.mkdtemp()

# 証跡の保存先。E2E_PLAN §0 の指定どおり aws-s3-sigv4-improvement/evidence/e2e/ 配下。
if evidence_dir is None:
    evidence_dir = os.environ.get(
        'E2E_EVIDENCE_DIR',
        os.path.join(os.pardir, 'aws-s3-sigv4-improvement', 'evidence', 'e2e'))
run_id = datetime.now().strftime('%Y%m%d-%H%M%S')
run_dir = os.path.abspath(os.path.join(evidence_dir, f'run-{run_id}'))
os.makedirs(run_dir, exist_ok=True)

# Playwright の成果物(video-N.webm / har.zip / console.log / last-screenshot.png /
# last-dom.html)は scripts/playwright.py:217-259 が default_result_path 配下へ書く。
# 既定のままだと work_dir に落ち、最終セルの `!rm -fr {work_dir}` ごと消える。
# har.zip は S-1(G-10)の判定に使うので、既定を run_dir にする。
if default_result_path is None:
    default_result_path = run_dir


def evidence(name):
    """証跡ファイルのパスを作る。名前には S-番号を含めること(E2E_PLAN §2)。"""
    return os.path.join(run_dir, name)


def record(name, text, quiet=False):
    """テキスト証跡を保存しつつ、ノートブックの出力にも残す。

    quiet=True のときは中身を印字しない(1500 件の一覧など、出力に流すと
    ノートブックが読めなくなるもの)。
    """
    path = evidence(name)
    with open(path, 'w') as f:
        f.write(text)
    print(f'--- {path} ({len(text)} bytes)')
    if not quiet:
        print(text)
    return path


print(run_dir)
print(work_dir)

# GakuNinRDM 総合テスト [Amazon S3 — SigV4 回帰]

- サブシステム名: 管理者 / ストレージ / WaterButler
- ページ/アドオン: 機関ストレージ (Amazon S3) および S3 アドオン
- 機能分類: SigV4 移行の回帰確認(実機)
- シナリオ名: AWS S3 を機関ストレージとアドオンの両方で使い、SigV4 署名・
  presigned URL リダイレクト・1000 件超の一覧・バージョン削除・intra copy/move
  が壊れていないことを確かめる
- **対象 PR: RDM-waterbutler#93(`feature/s3-sigv4`, head `cf2f87ab`)/ RDM-osf.io#746**
- 取る証跡: TEST_SPEC v1.8 の **P-6 / V-1 / I-2 / IF-3 / G-10** の実機証跡
- 用意するテストデータ: URL 一覧、アカウント(既存ユーザー1: GRDM)、管理者アカウント、
  AWS S3 のアクセスキー/シークレットキーと **3 つのバケット**
  (通常 / バージョニング有効 / `ap-northeast-1`)

## シナリオと受入項目の対応

| シナリオ | 受入項目 | 使うモード | 証跡 |
|---|---|---|---|
| S-0 環境確認 | IF-5 / R-5 | — | `S-0-jenkins-build.txt`(手動) / `S-0-params.txt` |
| S-1 機関ストレージ登録・アップロード・ダウンロード | IF-3 既定 / **G-10** | 機関ストレージ | `S-1-wb-responses.json` / `S-1-download-redirect.json` |
| S-2 明示リージョン(`ap-northeast-1`) | **IF-3** 明示 | 機関ストレージ | `S-2-wb-responses.json` |
| S-3 1000 件超フォルダ | **P-6** / R-6 | **S3 アドオン** | `S-3-listing-pages.json` / `S-3-cleanup.txt` |
| S-4 バージョニング有効バケットでの削除 | **V-1**(任意で V-2) | **S3 アドオン** | `S-4-versions-before.json` / `S-4-versions-after.json` |
| S-5 intra copy / move | **I-2** | **S3 アドオン** | `S-5-wb-responses.json` |
| S-6 CompleteMultipartUpload の Error Code 採取 | E-1 | — | `scripts/s3_complete_error_codes.py`(本ノートブックの外) |

## なぜ S-3 / S-4 / S-5 は機関ストレージではなく **アドオン**で行うのか

**AWS S3 を機関ストレージに登録した場合、ファイル操作は `s3` プロバイダに
直接は届かない。** 登録は `Region` を作るだけで、プロジェクトのファイルツリーは
`osfstorage` のままであり、`osfstorage` が内側で `s3` を呼ぶ形になる。根拠:

| 事実 | 位置 |
|---|---|
| 管理画面の保存は `Region` を作る(`provider: 's3'`, `type: Region.INSTITUTIONS`) | RDM-osf.io `admin/rdm_custom_storage_location/utils.py` `save_s3_credentials` |
| 機関の全 Node の **`osfstorage`** アドオンにその Region を割り当てる | 同 `update_nodes_storage` → `website/util/quota.py` `update_node_storage` |
| WB へは `osfstorage` の設定として Region の `waterbutler_settings` が渡る | `osf/models/files.py` `serialize_waterbutler_settings` |

その結果、`osfstorage` プロバイダの実装(RDM-waterbutler
`waterbutler/providers/osfstorage/provider.py`)により次のようになる。

| 操作 | 内側の `s3` に届くか | 位置 |
|---|---|---|
| アップロード | **届く**(本体バイト列は内側プロバイダに置かれる) | `upload` |
| ダウンロード | **届く**。`accept_url` がそのまま渡り、`s3` が presigned URL を返して WB が 302 する | `download` → `provider.download(**download_kwargs)` |
| 一覧(metadata) | **届かない**。OSF の DB を読むだけ | `metadata` → `make_signed_request` |
| 削除 | **届かない**。OSF に DELETE を投げるだけで `DeleteObject` は呼ばれない | `delete` |
| intra copy / move | **届かない**。OSF の `hooks/copy`・`hooks/move` を叩くだけ | `_do_intra_move_or_copy` |

さらに、1000 件超の追加読み込み(`next_token`)は fangorn が
**`s3` / `s3compat` / `s3compatinstitutions` / `s3compatsigv4` のときだけ**
親フォルダに `next_token` を積む実装になっている(RDM-osf.io
`website/static/js/fangorn.js` `_lazyLoadPreprocess`)。`osfstorage` は対象外。

したがって **P-6(一覧のページング)・V-1(バージョン削除)・I-2(intra copy/move)は
機関ストレージ経由では 1 行も実行されない**。これらを実機で踏むには
`s3` アドオンをプロジェクトに接続するしかない。E2E_PLAN v0.2 §1 は
「外部ストレージ(アドオン)モードでの網羅は別途」としているが、**網羅ではなく
当該 3 項目を実行できる唯一の経路**としてアドオンを使う。
(この判断は PHASE_E2E_PREP_REPORT.md にも記載した)

## リージョンの決まり方(IF-3 の検証方法)

**リージョンは UI のどこにも入力欄がない。** 管理画面の保存値は
`{folder, encrypt_uploads, bucket, provider, type}` だけで、リージョンを含まない
(`save_s3_credentials`)。WB 側は `S3Provider.__init__` で `self.region = None` と
し、最初のリクエストの前に `_check_region()` が `GetBucketLocation` を投げて
決める(`waterbutler/providers/s3/provider.py`)。`endpoint_url` は
`https://s3.<region>.amazonaws.com` に固定され、SigV4 はそのホストごと署名する。

よって **「明示リージョン」の検証は「`ap-northeast-1` に作ったバケットを登録する」
こと**で行う(S-2)。UI でリージョンを選ぶ手順は存在しない。

## 実行前に用意しておくこと

1. staging2 の **RDM-waterbutler が `cf2f87ab`(PR #93 head)**、
   **RDM-osf.io が PR #746 の head** に差し替わっていること。
   Jenkins `r-deploy` のビルドログを `S-0-jenkins-build.txt` に手で保存する(S-0)。
2. `.config.yaml`(リポジトリ直下、.gitignore 済み)に資格情報を書いておくか、
   実行時のプロンプトに入力する。**ノートブックにキーを書かない**。
3. AWS のバケットを 3 つ用意する。いずれも空であること。
   - 通常バケット(`s3_bucket`, `us-east-1`)
   - バージョニング**有効**バケット(`s3_bucket_versioned`, `us-east-1`)
   - `ap-northeast-1` のバケット(`s3_bucket_apne1`)
4. 機関ストレージの登録は**上書き**である(後述)。実施前に合意を取ること。

### 機関ストレージの登録は「追加」ではなく「上書き」である

機関ストレージは機関あたり 1 つしかなく、この操作は**現在の設定を破棄する**。
保存後に `update_nodes_storage` / `change_allowed_for_institutions` /
`add_node_settings_to_projects` が走り、影響はその機関の全 Node・全ユーザーに及ぶ
(RDM-osf.io `admin/rdm_custom_storage_location/views.py`)。
**「登録を外す」という操作は無い。** 別のプロバイダで上書きするのが唯一の戻し方。
後始末チェックリスト(末尾)を必ず最後まで消化すること。

In [ ]:
import asyncio
import importlib
import time

import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

### WaterButler への応答を記録する仕掛けを入れる

以降の操作で WaterButler が返した **HTTP ステータス**と**本文**を捕まえる。
S-2(署名エラーが無いこと)・S-3(`next_token` の連鎖)・S-5(copy/move の POST)の
判定根拠になる。

In [ ]:
wb_responses = []

# 一覧応答(?meta=)は本文を解析するので切らない。それ以外は証跡が膨らむので切る。
WB_BODY_LIMIT = 2000
WB_BODY_LIMIT_META = 4 * 1024 * 1024


async def _attach_recorder(page):
    async def _on_response(response):
        url = response.url
        if '/v1/resources/' not in url and '/wb/' not in url:
            return
        entry = {
            'status': response.status,
            'method': response.request.method,
            'url': url,
            'at': datetime.now().isoformat(),
        }
        limit = WB_BODY_LIMIT_META if 'meta=' in url else WB_BODY_LIMIT
        try:
            entry['body'] = (await response.text())[:limit]
        except Exception as e:
            entry['body'] = f'<unavailable: {e!r}>'
        wb_responses.append(entry)

    page.on('response', lambda r: asyncio.ensure_future(_on_response(r)))


async def _step(page):
    await _attach_recorder(page)

await run_pw(_step)


def _redact(text):
    """署名と資格情報を伏せる。証跡に出す前に必ず通す。"""
    import re
    out = text
    for secret in [s3_secret_key, s3_access_key]:
        if secret:
            out = out.replace(secret, '***')
    out = re.sub(r'(X-Amz-(?:Signature|Credential|Security-Token)=)[^&\s"\\]+',
                 r'\1***', out)
    out = re.sub(r'AKIA[0-9A-Z]{12,}', 'AKIA***', out)
    return out


def dump_wb_responses(name, keep=False, quiet=True):
    """直近の操作で観測した WB 応答を証跡に落とす。"""
    text = _redact(json.dumps(wb_responses, indent=2, ensure_ascii=False))
    assert 'AKIA' not in text.replace('AKIA***', ''), '証跡にアクセスキーが残っている'
    path = record(name, text, quiet=quiet)
    if not keep:
        wb_responses.clear()
    return path


def wb_failures(entries=None):
    """4xx / 5xx と、署名系の文言を含む応答を拾う。"""
    entries = wb_responses if entries is None else entries
    bad_words = ('SignatureDoesNotMatch', 'AuthorizationHeaderMalformed',
                 'PermanentRedirect', 'InvalidAccessKeyId')
    out = []
    for e in entries:
        body = e.get('body') or ''
        if e['status'] >= 400 or any(w in body for w in bad_words):
            out.append(e)
    return out


async def open_storage(page, storage_name):
    """ファイルツリーのストレージノードを開き、アップロードボタンが出る状態にする。"""
    await page.locator(f'//*[contains(text(), "{storage_name}")]').first.click()
    await asyncio.sleep(2)
    upload_btn = page.locator('//i[contains(@class, "fa-upload")]/../*[text() = "アップロード"]')
    await expect(upload_btn).to_be_visible(timeout=transition_timeout)


async def clear_growls(page):
    """残っている growl を消す。

    growl(`[data-growl="container"]`)は自動では消えないことがあり、前の手順の
    growl が残っていると「今の操作の結果」と区別がつかない。判定の直前に消す。
    """
    await page.evaluate(
        '() => { document.querySelectorAll(\'[data-growl="container"]\')'
        '.forEach(function(e) { e.remove(); }); }')


async def growl_messages(page):
    """今表示されている growl の本文。判定の根拠として証跡に残す。"""
    return await page.locator('//*[@data-growl = "message"]').all_inner_texts()


async def uploading_rows(page, filename=None):
    """アップロード中(progressbar 付きの仮行)の数。"""
    if filename is None:
        return await page.locator('//*[@role = "progressbar"]').count()
    return await page.locator(
        f'//*[text() = "{filename}"]/../following-sibling::*//*[@role = "progressbar"]'
    ).count()


async def wait_for_upload_finished(page, filename, timeout=None):
    """成否を問わず「アップロードが終わった」ことを待ち、経過秒を返す。

    fangorn は成功時も失敗時も仮行を消す。**仮行が残っている = XHR がまだ
    返っていない。** Dropzone は parallelUploads: 1 なので、終わっていない行を
    残したまま次の手順に進むと次のアップロードはキューで止まる。
    """
    timeout = transition_timeout * 10 if timeout is None else timeout
    started = time.time()
    deadline = started + timeout / 1000
    while time.time() < deadline:
        if await uploading_rows(page, filename) == 0:
            return time.time() - started
        await asyncio.sleep(2)
    raise AssertionError(
        f'{filename} のアップロードが {timeout / 1000:.0f} 秒で終わらない')


def s3_client(region):
    """boto3 の S3 クライアント。資格情報はノートブックのパラメータから取る。"""
    import boto3
    from botocore.config import Config
    return boto3.client(
        's3',
        region_name=region,
        aws_access_key_id=s3_access_key,
        aws_secret_access_key=s3_secret_key,
        config=Config(signature_version='s3v4', retries={'max_attempts': 5}),
    )

## S-0: 環境確認(IF-5 / R-5)

### 手で保存するもの — `S-0-jenkins-build.txt`

Jenkins の `r-deploy` ジョブのビルドログから、次の行をコピーして
証跡ディレクトリに `S-0-jenkins-build.txt` として保存する。
**ビルドログを取らずに「差し替えたはず」で進めないこと。**
このジョブは WB だけでなく osf.io / CAS / MFR など 8 リポジトリを同時に
再ビルドするため、直前のビルドが誰の何だったかは追跡が要る。

1. stage `build RDM-waterbutler` の
   `Checking out Revision cf2f87ab...`
   — PR RCOSDP/RDM-waterbutler#93 の head をビルドしたことの証明
2. stage `build RDM-osf.io` の
   `Checking out Revision <#746 の head SHA>`
   — PR RCOSDP/RDM-osf.io#746 の head をビルドしたことの証明。
     SHA は実施者が PR 画面で確認して記録する
3. デプロイ段の
   `helm upgrade -i rdm-wb ... --set image.tag=<TAG>` と
   `helm upgrade -i rdm-osf ... --set image.tag=<TAG>`
   — そのビルドが実際に staging2 に載ったことの証明

`wb_deployed_sha` / `osf_deployed_sha` に手で控えて、次のセルで証跡に残す。

In [ ]:
# S-0: 実行時パラメータの記録。**キーは書かない**。
wb_deployed_sha = 'cf2f87ab'   # PR RCOSDP/RDM-waterbutler#93 head
osf_deployed_sha = None        # PR RCOSDP/RDM-osf.io#746 head。実施者が記入する

s0 = {
    'run_id': run_id,
    'started_at': datetime.now().isoformat(),
    'admin_rdm_url': admin_rdm_url,
    'rdm_url': rdm_url,
    'wb_pr': 'RCOSDP/RDM-waterbutler#93',
    'wb_deployed_sha': wb_deployed_sha,
    'osf_pr': 'RCOSDP/RDM-osf.io#746',
    'osf_deployed_sha': osf_deployed_sha,
    'target_organization': target_organization,
    'institutional_storage_name': institutional_storage_name,
    'addon_storage_id': addon_storage_id,
    'buckets': {
        'default': {'name': s3_bucket, 'region': s3_region_default,
                    'used_by': ['S-1', 'S-3', 'S-5']},
        'apne1': {'name': s3_bucket_apne1, 'region': s3_region_explicit,
                  'used_by': ['S-2']},
        'versioned': {'name': s3_bucket_versioned, 'region': s3_region_default,
                      'used_by': ['S-4']},
    },
    'switches': {
        'run_apne1': run_apne1,
        'run_toomany': run_toomany,
        'run_versioned': run_versioned,
        'run_versioned_bulk': run_versioned_bulk,
        'run_intra': run_intra,
        'toomany_count': toomany_count,
    },
    'rdm_project_prefix': rdm_project_prefix,
}
_text = json.dumps(s0, indent=2, ensure_ascii=False)
assert s3_access_key not in _text and s3_secret_key not in _text, \
    'S-0-params.txt に資格情報が混ざっている'
record('S-0-params.txt', _text)
if osf_deployed_sha is None:
    print('\n[!] osf_deployed_sha が未記入。PR #746 の head SHA を上のセルに記入して再実行すること')

In [ ]:
# S-0: バケットの前提を boto3 で確認する(空であること・リージョン・バージョニング)
def bucket_region(bucket):
    loc = s3_client(s3_region_default).get_bucket_location(Bucket=bucket)
    # GetBucketLocation は us-east-1 を None(旧 API では '') で返す
    return loc.get('LocationConstraint') or 'us-east-1'


def bucket_state(bucket, region):
    c = s3_client(region)
    listed = c.list_objects_v2(Bucket=bucket, MaxKeys=5)
    ver = c.get_bucket_versioning(Bucket=bucket)
    return {
        'bucket': bucket,
        'declared_region': region,
        'GetBucketLocation': bucket_region(bucket),
        'KeyCount': listed.get('KeyCount', 0),
        'sample_keys': [o['Key'] for o in listed.get('Contents', [])],
        'Versioning': ver.get('Status', 'Disabled'),
    }


_states = [bucket_state(s3_bucket, s3_region_default)]
if run_apne1:
    _states.append(bucket_state(s3_bucket_apne1, s3_region_explicit))
if run_versioned:
    _states.append(bucket_state(s3_bucket_versioned, s3_region_default))

for _s in _states:
    assert _s['GetBucketLocation'] == _s['declared_region'], (
        f"{_s['bucket']} の所在は {_s['GetBucketLocation']} で、"
        f"宣言した {_s['declared_region']} と違う。"
        'WB はバケットの所在でエンドポイントを決めるので、ここが食い違うと'
        'IF-3 の判定にならない')
if run_versioned:
    assert _states[-1]['Versioning'] == 'Enabled', \
        's3_bucket_versioned のバージョニングが有効になっていない(S-4 の前提)'

record('S-0-buckets.json', json.dumps(_states, indent=2, ensure_ascii=False))

## S-1: 機関ストレージ登録 + 基本操作(IF-3 既定 / G-10)

### Part 1: 管理者画面での機関ストレージ設定

`s3_bucket`(`us-east-1`)を機関ストレージとして登録する。
WB は `GetBucketLocation` の結果でエンドポイントを決めるので、この登録が
「既定リージョン」の検証にあたる。

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url)

    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

### ログイン情報を用いてGakuNin RDM管理者画面にログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。

In [ ]:
import urllib.parse


async def admin_login(page):
    email_field = page.locator('#id_email')
    if await email_field.count() > 0 and await email_field.is_visible():
        # Django メール/パスワードフォーム（enable_form=True の環境）
        await email_field.fill(admin_username)
        await page.locator('#id_password').fill(admin_password)
        await page.locator('//form//button[@type="submit"]').click()
    else:
        # entityID 直接指定で Shibboleth ログインを開始（Embedded DS バイパス）
        if admin_idp_entity_id is None:
            raise ValueError(
                'admin_idp_entity_id が未設定です。管理者ログインがメール/パスワード形式でない場合は '
                '.config.yaml または本ノートブックのパラメータで対象環境のIdP entityIDを指定してください。'
            )
        base = admin_rdm_url.rstrip('/')
        url = (base + '/Shibboleth.sso/Login'
               '?entityID=' + urllib.parse.quote(admin_idp_entity_id, safe='')
               + '&target=' + urllib.parse.quote(base + '/account/shib-login', safe=''))
        await page.goto(url)
        print('[1] after goto:', page.url)

        title = await page.title()
        assert 'Unknown Identity Provider' not in title, (
            f'SPが entityID を認識していません: {admin_idp_entity_id}')

        username = page.locator('#username')
        j_username = page.locator('input[name="j_username"]')
        consent = page.locator('#_shib_idp_doNotRememberConsent')
        logout_link = page.locator('//*[@href="/account/logout/"]')

        # IdPログインフォーム / 同意画面 / 既ログイン のいずれかを待つ
        await expect(username.or_(j_username).or_(consent).or_(logout_link).first
                     ).to_be_visible(timeout=transition_timeout)

        if await username.count() > 0 and await username.is_visible():
            await username.fill(admin_username)
            await page.locator('#password').fill(admin_password)
            await page.locator('//button[@type="submit"] | //input[@type="submit"]').first.click()
            print('[2] after credential submit:', page.url)
        elif await j_username.count() > 0 and await j_username.is_visible():
            await j_username.fill(admin_username)
            await page.locator('input[name="j_password"]').fill(admin_password)
            await page.locator('//button[@type="submit"] | //input[@type="submit"]').first.click()
            print('[2] after credential submit (j_username):', page.url)
        else:
            print('[2] ログインフォームなし（既ログイン or 同意画面）:', page.url)

        # ログイン失敗の検出（IdPのエラー表示）
        await asyncio.sleep(2)
        err = page.locator('.form-error, .output--error, p.form-element.form-error')
        if await err.count() > 0 and await err.first.is_visible():
            raise AssertionError('IdPログイン失敗: ' + (await err.first.text_content()).strip())

        # 同意画面（出ない場合はスキップ）
        try:
            await expect(consent).to_be_visible(timeout=15000)
            await consent.click()
            proceed = page.locator('//*[@name="_eventId_proceed"]')
            await expect(proceed).to_be_enabled()
            await proceed.click()
            print('[3] consent done:', page.url)
        except AssertionError:
            print('[3] 同意画面はスキップ:', page.url)

    # ログアウトボタンまたはlogoutリンクが表示されること（日本語/英語環境両対応）
    logout_btn = page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')
    logout_link2 = page.locator('//*[@href="/account/logout/"]')
    try:
        await expect(logout_btn.or_(logout_link2)).to_be_visible(timeout=transition_timeout)
    except AssertionError:
        print('[4] 最終確認失敗。現在地:', page.url, '/', await page.title())
        body = await page.evaluate('() => document.body.innerText.substring(0, 800)')
        print(body)
        raise
    print('[4] 管理者画面ログイン成功:', page.url)


await run_pw(admin_login)

### 機関ストレージの登録手順を関数にする

S-1(既定リージョン)と S-2(`ap-northeast-1`)で同じ手順を 2 回使うので関数にする。
元の `テスト手順-管理者機能-S3-機関ストレージ.ipynb` の Part 1(セル 12〜28)を
そのまま関数へ移したもの。

**この登録は上書きである**(セル [5] 参照)。`register_institutional_s3` を呼ぶたびに
その機関の機関ストレージ設定が置き換わる。

In [ ]:
async def goto_institutional_storage(page):
    await page.goto(admin_rdm_url.rstrip('/') + '/custom_storage_location/institutional_storage/')
    # サイドメニューの「機関ストレージ」をクリック（「機関ストレージのクォータ」と区別）
    link = page.locator('//a[contains(@href, "institutional_storage") and not(contains(@href, "quota"))]').first
    await expect(link).to_be_visible(timeout=transition_timeout)
    await link.click()
    # 機関ストレージ設定画面が表示されること（ラジオボタンの存在で確認）
    await expect(page.locator('//input[@type="radio"]').first).to_be_visible(timeout=transition_timeout)


async def select_institution(page):
    """機関のリストから target_organization を選ぶ。

    単一機関の環境では機関リストが出ず、いきなり設定画面になる。その場合は
    何もしない(元ノートブックではこのセルはコメントアウトされていた)。
    """
    if target_organization is None:
        print('target_organization 未指定。機関リストの選択はスキップ')
        return
    link = page.locator(f'//a[text() = "{target_organization}"]')
    if await link.count() == 0:
        print(f'機関リストに "{target_organization}" が見つからない。既に設定画面とみなす')
        return
    await link.first.click()
    await expect(page.locator('//h2[contains(text(), "Institutional Storage")]')
                 ).to_be_visible(timeout=transition_timeout)


async def capture_institutional_storage_state(page, name):
    """登録前/後の選択状態を証跡に残す。後始末で戻すときの手がかりになる。"""
    state = await page.evaluate(
        '() => { var r = document.querySelector("input[name=options]:checked");'
        ' var n = document.getElementById("storage_name");'
        ' return {selected: r ? r.value : "none", storage_name: n ? n.value : null}; }')
    await page.screenshot(path=evidence(name + '.png'), full_page=True)
    record(name + '.txt', json.dumps(state, indent=2, ensure_ascii=False))
    return state

In [ ]:
async def register_institutional_s3(page, bucket, label):
    """Amazon S3 を機関ストレージとして登録する。**既存設定の上書きである。**

    :param bucket: 登録するバケット名
    :param label: 証跡のファイル名に使う識別子(例 'S-1')
    """
    await goto_institutional_storage(page)
    await select_institution(page)

    # Amazon S3 のラジオボタンを選択
    # （既定で別プロバイダ(例: s3compatsigv4)が選択済みの場合があるため、check + 選択状態を検証する）
    radio = page.locator('//input[@type="radio" and @value="s3"]')
    await expect(radio).to_be_visible(timeout=transition_timeout)
    await radio.scroll_into_view_if_needed()
    try:
        await radio.check()
    except Exception:
        await radio.check(force=True)
    await expect(radio).to_be_checked()
    selected = await page.evaluate(
        '() => { const r = document.querySelector("input[name=options]:checked");'
        ' return r ? r.value : "none"; }')
    assert selected == 's3', f'Amazon S3 を選択できていません: {selected}'

    # storage_name フィールドの状態を確認し、必要なら入力する
    storage_name = page.locator('#storage_name')
    sn_visible = await storage_name.is_visible()
    sn_enabled = await storage_name.is_enabled() if sn_visible else False
    sn_value = await storage_name.input_value() if sn_visible else ''
    print(f'storage_name: visible={sn_visible}, enabled={sn_enabled}, value="{sn_value}"')
    if sn_visible and sn_enabled and not sn_value.strip():
        await storage_name.fill(institutional_storage_name)

    # Save(保存) ボタンをクリック
    save_btn = page.locator('//button[@type="submit" and contains(@class, "btn-success")]').first
    await save_btn.scroll_into_view_if_needed()
    await save_btn.click()
    await asyncio.sleep(2)

    confirm_text = page.locator('#bbConfirmText')
    s3_modal = page.locator('#s3_modal')

    # 確認ダイアログ(bootbox)が出た場合は確認文字列を入力して変更を確定する
    try:
        await expect(confirm_text).to_be_visible(timeout=10000)
        confirm_strong = page.locator('//div[contains(@class, "bootbox-body")]//strong')
        confirmation_string = await confirm_strong.text_content()
        print(f'Confirmation string: {confirmation_string}')
        await confirm_text.fill(confirmation_string)
        await page.locator(
            '//div[contains(@class, "modal") and contains(@class, "bootbox")]'
            '//button[contains(@class, "btn-danger")]').click()
        print('確認ダイアログを処理しました')
    except Exception:
        print('確認ダイアログは表示されませんでした')

    # Amazon S3 認証モーダルが開くこと。**開かなければ登録できていない。**
    # 元ノートブックは「既設定とみなしてスキップ」していたが、本テストは
    # バケットを差し替えること自体が手順なので、開かないなら失敗として扱う。
    await expect(s3_modal).to_be_visible(timeout=20000)

    await page.locator('#s3_access_key').fill(s3_access_key)
    await page.locator('#s3_secret_key').fill(s3_secret_key)
    await page.locator('#s3_bucket').fill(bucket)
    # keyupイベントを発火させてバリデーションをトリガー
    await page.evaluate(
        '() => { document.querySelectorAll("#s3_modal input")'
        '.forEach(el => el.dispatchEvent(new Event("keyup", { bubbles: true }))); }')
    await asyncio.sleep(1)

    # Connect(接続テスト)。Save が有効になるのが接続成功の証。
    connect_btn = page.locator('#s3_connect')
    await expect(connect_btn).to_be_enabled(timeout=transition_timeout)
    await connect_btn.click()
    save_modal_btn = page.locator('#s3_save')
    await expect(save_modal_btn).to_be_enabled(timeout=transition_timeout)

    msg = await page.evaluate(
        '() => document.getElementById("s3_message")'
        ' ? document.getElementById("s3_message").innerText : "no message element"')
    print('server message:', msg)
    record(f'{label}-connect-message.txt',
           f'bucket={bucket}\nserver message={msg}\n')

    await save_modal_btn.click()
    await expect(s3_modal).to_be_hidden(timeout=transition_timeout)
    await asyncio.sleep(2)
    await capture_institutional_storage_state(page, f'{label}-institutional-storage-after')
    print(f'{label}: {bucket} を機関ストレージとして登録した')

In [ ]:
# 登録前の状態を証跡に残してから、S-1 の登録を行う
async def _step(page):
    await goto_institutional_storage(page)
    await select_institution(page)
    before = await capture_institutional_storage_state(
        page, 'S-0-institutional-storage-before')
    print('登録前:', before)

await run_pw(_step)

In [ ]:
async def _step(page):
    await register_institutional_s3(page, s3_bucket, 'S-1')

await run_pw(_step)

### 管理者画面からログアウトする

ログアウトが完了すること

In [ ]:
async def admin_logout(page):
    logout_btn = page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')
    logout_link = page.locator('//*[@href="/account/logout/"]')
    if await logout_btn.count() > 0:
        await logout_btn.click()
    else:
        await logout_link.click()

    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

await run_pw(admin_logout)

### Part 2: ユーザ画面でのプロジェクト作成・アップロード・ダウンロード

`institutional_storage_name` のノードがファイルツリーに出ること(= 機関ストレージが
反映されたこと)、アップロードとダウンロードが通ることを確認する。
ダウンロードの 302 と presigned URL は、Playwright を閉じたあとに HAR を読んで
判定する(末尾の S-1(G-10) のセル)。

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)


async def _step(page):
    await page.goto(rdm_url)
    await page.wait_for_load_state('networkidle')

    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count() > 0 and await consent_button.is_visible():
        await consent_button.click()
        await page.wait_for_load_state('networkidle')

    await scripts.grdm.expect_anonymous_toppage(
        page, idp_name_1, transition_timeout=transition_timeout)

await run_pw(_step)

In [ ]:
async def _step(page):
    await scripts.grdm.login(
        page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout
    )
    await expect(page.locator('//*[@data-test-create-project-modal-button]')
                 ).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

In [ ]:
async def open_project(page, project_name):
    """ダッシュボードから対象プロジェクトを開いてファイルタブを表示する。"""
    await page.goto(rdm_url)
    await expect(page.locator('//*[@data-test-create-project-modal-button]')
                 ).to_be_visible(timeout=transition_timeout)
    await page.locator(
        f'//*[@data-test-dashboard-item-title and text()="{project_name}"]').click()
    await expect(page.locator('#projectNavFiles')).to_be_visible(timeout=transition_timeout)


async def open_files_tab(page):
    await page.locator('#projectNavFiles a').click()
    await expect(page.locator('//*[@id = "treeGrid"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(3)


async def create_project_and_open_files(page, project_name):
    await page.goto(rdm_url)
    await expect(page.locator('//*[@data-test-create-project-modal-button]')
                 ).to_be_visible(timeout=transition_timeout)
    await scripts.grdm.ensure_project_exists(page, project_name, transition_timeout)
    await open_project(page, project_name)
    await open_files_tab(page)


rdm_project_inst = f'{rdm_project_prefix}-institutional'

async def _step(page):
    await create_project_and_open_files(page, rdm_project_inst)

await run_pw(_step)

In [ ]:
# S-1 用のテストファイル。バケット名は書かない(証跡は S-0-params.txt にある)
s1_file_name = 'S-1-institutional.txt'
s1_file_path = os.path.join(work_dir, s1_file_name)
with open(s1_file_path, 'w') as f:
    f.write('Institutional Storage Test - Amazon S3 (SigV4)\n')
    f.write(f'run_id: {run_id}\n')
    f.write(f'Created at: {datetime.now().isoformat()}\n')

print(s1_file_path, os.path.getsize(s1_file_path), 'bytes')

In [ ]:
async def _step(page):
    await open_storage(page, institutional_storage_name)
    await scripts.grdm.upload_file(page, s1_file_path)
    await wait_for_upload_finished(page, s1_file_name)
    await expect(page.locator(f'//*[contains(text(), "{s1_file_name}")]')
                 ).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

In [ ]:
async def _step(page):
    # ファイル詳細ページが表示されること（ファイル名ヘッダーの拡張子で確認）
    await grdm.get_select_file_title_locator(page, s1_file_name).click(timeout=transition_timeout)
    await asyncio.sleep(3)
    body, ext = os.path.splitext(s1_file_name)
    await expect(page.locator(f'//h2[contains(text(), "{body}")]//*[@id = "file-ext"]')
                 ).to_have_text(ext, timeout=transition_timeout)

await run_pw(_step)

### S-1 (G-10): ダウンロードする

ファイル詳細画面の「ダウンロード」を押し、ブラウザのダウンロードイベントを待つ。

WB は `accept_url` が真のとき presigned URL を返し、ハンドラが 302 でそこへ飛ばす
(`waterbutler/server/api/v1/provider/metadata.py` の `get_file` →
`if isinstance(stream, str): return self.redirect(stream)`)。
**302 と `Location` の中身は HAR でしか取れない**ので、判定は末尾のセルで行う
(HAR は `finish_pw_context` のときに書き出される)。

In [ ]:
s1_download_path = None

async def _step(page):
    global s1_download_path
    download_dir = os.path.join(work_dir, 'downloaded')
    os.makedirs(download_dir, exist_ok=True)

    btn = page.locator(
        '//i[contains(@class, "fa-download")]/..'
        '/*[text() = "ダウンロード" or text() = "Download"]')
    if await btn.count() == 0:
        btn = page.locator('//a[contains(@href, "?action=download")]')
    async with page.expect_download(timeout=transition_timeout * 3) as download_info:
        await btn.first.click()
    download = await download_info.value
    s1_download_path = os.path.join(download_dir, download.suggested_filename)
    await download.save_as(s1_download_path)
    print('downloaded:', s1_download_path, os.path.getsize(s1_download_path), 'bytes')

await run_pw(_step)

with open(s1_file_path) as _a, open(s1_download_path) as _b:
    assert _a.read() == _b.read(), 'ダウンロードした内容がアップロードしたものと違う'
print('S-1: アップロードした内容と一致')

In [ ]:
# S-1 の判定: WB 応答に 4xx/5xx も署名系の文言も無いこと
_bad = wb_failures()
record('S-1-verdict.txt',
       f'wb_responses={len(wb_responses)}\n'
       f'failures={len(_bad)}\n'
       + _redact(json.dumps(_bad, indent=2, ensure_ascii=False)))
dump_wb_responses('S-1-wb-responses.json')
assert not _bad, 'S-1 で WB がエラーを返している(上の S-1-verdict.txt を見ること)'

## S-2: 明示リージョン(`ap-northeast-1`)(IF-3)

`s3_bucket_apne1` を同じ機関に**上書き**登録し、アップロードとダウンロードを
やり直す。WB は `GetBucketLocation` で `ap-northeast-1` を得て
`https://s3.ap-northeast-1.amazonaws.com` を署名対象ホストにする。

**判定**: WB 応答に 4xx/5xx が無く、本文に `SignatureDoesNotMatch` /
`AuthorizationHeaderMalformed` / `PermanentRedirect` を含まないこと。
`PermanentRedirect` は「署名したホストと送ったホストが食い違っている」ときに
S3 が返すコードで、エンドポイント固定が壊れたときの典型的な症状である。

別プロジェクトを作って実施する。S-1 のプロジェクトは `us-east-1` のバケットに
本体を置いた版を持っており、登録を差し替えたあとに同じプロジェクトを使うと
どちらのバケットを見ているのか証跡上あいまいになるため。

In [ ]:
rdm_project_apne1 = f'{rdm_project_prefix}-apne1'

if run_apne1:
    async def _step(page):
        await page.goto(admin_rdm_url)
        await admin_login(page)
        await register_institutional_s3(page, s3_bucket_apne1, 'S-2')
        await admin_logout(page)

    await run_pw(_step)
else:
    print('run_apne1=False のためスキップ')

In [ ]:
s2_file_name = 'S-2-apne1.txt'
s2_file_path = os.path.join(work_dir, s2_file_name)

if run_apne1:
    with open(s2_file_path, 'w') as f:
        f.write('Explicit region test - ap-northeast-1 (SigV4)\n')
        f.write(f'run_id: {run_id}\n')
        f.write(f'Created at: {datetime.now().isoformat()}\n')

    async def _step(page):
        await create_project_and_open_files(page, rdm_project_apne1)
        await open_storage(page, institutional_storage_name)
        await scripts.grdm.upload_file(page, s2_file_path)
        await wait_for_upload_finished(page, s2_file_name)
        await expect(page.locator(f'//*[contains(text(), "{s2_file_name}")]')
                     ).to_be_visible(timeout=transition_timeout)

    await run_pw(_step)
else:
    print('run_apne1=False のためスキップ')

In [ ]:
s2_download_path = None

if run_apne1:
    async def _step(page):
        global s2_download_path
        await grdm.get_select_file_title_locator(page, s2_file_name).click(
            timeout=transition_timeout)
        await asyncio.sleep(3)
        download_dir = os.path.join(work_dir, 'downloaded')
        os.makedirs(download_dir, exist_ok=True)
        btn = page.locator(
            '//i[contains(@class, "fa-download")]/..'
            '/*[text() = "ダウンロード" or text() = "Download"]')
        if await btn.count() == 0:
            btn = page.locator('//a[contains(@href, "?action=download")]')
        async with page.expect_download(timeout=transition_timeout * 3) as download_info:
            await btn.first.click()
        download = await download_info.value
        s2_download_path = os.path.join(download_dir, download.suggested_filename)
        await download.save_as(s2_download_path)

    await run_pw(_step)

    with open(s2_file_path) as _a, open(s2_download_path) as _b:
        assert _a.read() == _b.read(), 'S-2 のダウンロード内容が一致しない'
    print('S-2: アップロードした内容と一致')
else:
    print('run_apne1=False のためスキップ')

In [ ]:
if run_apne1:
    # バケットに実体が入ったことを boto3 でも確かめる(osfstorage は内容アドレスで
    # 置くのでキー名はハッシュになる。件数と所在だけ見る)
    _c = s3_client(s3_region_explicit)
    _listed = _c.list_objects_v2(Bucket=s3_bucket_apne1, MaxKeys=10)
    _bad = wb_failures()
    _verdict = {
        'bucket': s3_bucket_apne1,
        'GetBucketLocation': bucket_region(s3_bucket_apne1),
        'KeyCount': _listed.get('KeyCount', 0),
        'wb_responses': len(wb_responses),
        'failures': _bad,
    }
    record('S-2-verdict.json', _redact(json.dumps(_verdict, indent=2, ensure_ascii=False)))
    dump_wb_responses('S-2-wb-responses.json')
    assert not _bad, 'S-2 で WB がエラーを返している(署名エラーの可能性)'
    assert _listed.get('KeyCount', 0) > 0, \
        f'{s3_bucket_apne1} にオブジェクトが無い。別のバケットに書かれている'
    print('S-2: 署名エラーなし。ap-northeast-1 のバケットに実体が入っている')
else:
    print('run_apne1=False のためスキップ')

## S-3 〜 S-5 の前に: プロジェクトに **S3 アドオン**を接続する

セル [5] の表のとおり、一覧・削除・intra copy/move は機関ストレージ(= `osfstorage`
経由)では `s3` プロバイダに届かない。ここから先はプロジェクトに
`Amazon S3` アドオンを接続し、ファイルツリーの `Amazon S3` ノードで操作する。

接続手順は `取りまとめ-S3共通.ipynb` のセル 27〜45 と同じ。関数にして
S-3/S-5 用(通常バケット)と S-4 用(バージョニング有効バケット)で 2 回使う。

In [ ]:
async def connect_s3_addon(page, project_name, bucket):
    """プロジェクトに S3 アドオンを接続し、ファイルページを開く。

    取りまとめ-S3共通.ipynb のセル 27〜45 と同じ手順。
    アカウント(アクセスキー)はユーザー単位に保持されるので、2 回目以降は
    「アカウントに接続する」ではなく既存アカウントの選択になる。両方に対応する。
    """
    await open_project(page, project_name)

    # アドオンタブ → 有効にする → 確認
    await page.locator('//a[text() = "アドオン"]').click()
    enable_link = page.locator(
        f'//div[@full_name = "{addon_storage_name}"]//descendant::a[text() = "有効にする"]')
    await expect(enable_link).to_be_visible(timeout=transition_timeout)
    await enable_link.click()
    confirm = page.locator('//button[@data-bb-handler = "confirm"]')
    await expect(confirm).to_be_visible(timeout=transition_timeout)
    await confirm.click()
    await asyncio.sleep(2)

    # 資格情報の入力欄が出たら入力して保存する(初回のみ)
    input_creds = page.locator(f'#{addon_storage_id}InputCredentials')
    connect_link = page.locator(
        f'//img[@src = "/static/addons/{addon_storage_id}/comicon.png"]/..'
        f'//a[contains(text(), "アカウントに接続する")]')
    import_link = page.locator('//a[contains(text(), "プロフィールからアカウントをインポート")]')

    if await connect_link.count() > 0:
        await connect_link.first.click()
        await expect(input_creds.locator('.btn-success')).to_be_enabled(
            timeout=transition_timeout)
        await asyncio.sleep(1)
        await page.locator(
            f'//*[@id = "{addon_storage_id}InputCredentials"]//input[@id = "access_key"]'
        ).fill(s3_access_key)
        await page.locator(
            f'//*[@id = "{addon_storage_id}InputCredentials"]//input[@id = "secret_key"]'
        ).fill(s3_secret_key)
        await input_creds.locator('.btn-success').click()
    elif await import_link.count() > 0:
        await import_link.first.click()
        await asyncio.sleep(2)

    # 接続 → バケット選択 → 保存
    connect_btn = page.locator('//button[text() = "接続"]')
    await expect(connect_btn).to_be_enabled(timeout=transition_timeout)
    await asyncio.sleep(1)
    await connect_btn.click()
    await expect(page.locator('//button[@id = "newBucket"]')).to_be_visible(
        timeout=transition_timeout)
    await page.locator(
        f'//span[text() = "{bucket}"]/../../..//input[@type = "radio"]').click()
    # delay を入れないと「正常にリンクされました」が出ない(取りまとめ NB セル 43 の注記)
    await asyncio.sleep(1)
    await page.locator(
        f'//*[@class = "{addon_storage_id}-confirm-selection"]//input[@value = "保存"]').click()
    await expect(page.locator(
        f'//*[@id = "{addon_storage_id}Scope"]/*[contains(@class, "help-block")]'
        f'//*[contains(text(), "正常にリンクされました")]')).to_be_visible(timeout=30000)

    # ファイルページへ
    try:
        await page.get_by_role('link', name='ファイルページ').click()
    except Exception:
        await page.locator('//*[@id = "projectNavFiles"]//a').click()
    await expect(page.locator('//*[@id = "treeGrid"]')).to_be_visible(
        timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, addon_storage_name)
                 ).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(2)
    print(f'{project_name}: {addon_storage_name} アドオンを接続した')

In [ ]:
def listing_pages(entries=None, path_contains=None):
    """WB 応答から「フォルダ一覧(?meta=)」だけを抜き、next_token の連鎖に直す。

    - 要求側の next_token は URL のクエリから読む
      (fangorn.js `_fangornResolveLazyLoad` が `next_token` を付ける)
    - 応答側の next_token は本文のトップレベル `next_token`
      (waterbutler/server/api/v1/provider/metadata.py `get_folder`)
    """
    from urllib.parse import urlparse, parse_qs
    entries = wb_responses if entries is None else entries
    pages = []
    for e in entries:
        if e['method'] != 'GET' or 'meta=' not in e['url']:
            continue
        parsed = urlparse(e['url'])
        if path_contains and path_contains not in parsed.path:
            continue
        q = parse_qs(parsed.query)
        page = {
            'url_path': parsed.path,
            'request_next_token': q.get('next_token', [None])[0],
            'status': e['status'],
            'at': e['at'],
        }
        try:
            body = json.loads(e['body'])
        except Exception:
            page['parse_error'] = True
            page['names'] = []
            page['response_next_token'] = None
        else:
            data = body.get('data', []) if isinstance(body, dict) else []
            page['count'] = len(data)
            page['names'] = [d.get('attributes', {}).get('name') for d in data]
            page['response_next_token'] = (
                body.get('next_token') if isinstance(body, dict) else None)
        pages.append(page)
    return pages

## S-3: 1000 件超フォルダの一覧(P-6 / R-6)

S3 の ListObjectsV2 は 1 回で最大 1000 件しか返さない。WB は
`get_folder_metadata(..., next_token=...)` で続きを取り、ハンドラが応答の
末尾に `next_token` を載せる。fangorn はスクロールが底に着くたびに
`next_token` 付きで再要求する(`fetchData` / `handleScroll`)。

**判定**(指示 §1.5):

1. `toomany/` の中身が `0001` 〜 `{toomany_count}` まで**重複・欠落なく**表示される
2. 一覧応答の `next_token` が連鎖し、**最終ページで空**になる

`toomany/` の中身は boto3 の `put_object`(空本文)で作る。UI からの 1500 回
アップロードは時間がかかるうえ、検証したいのは一覧のページングだけなので
直接置く。

In [ ]:
# S-3: toomany/0001 〜 NNNN を boto3 で作る
s3_addon_project = f'{rdm_project_prefix}-addon'
toomany_names = ['{0:04d}'.format(i + 1) for i in range(toomany_count)]

if run_toomany:
    _c = s3_client(s3_region_default)
    _existing = set()
    _paginator = _c.get_paginator('list_objects_v2')
    for _p in _paginator.paginate(Bucket=s3_bucket, Prefix=toomany_prefix):
        for _o in _p.get('Contents', []):
            _existing.add(_o['Key'])

    _t0 = time.time()
    _made = 0
    for _name in toomany_names:
        _key = toomany_prefix + _name
        if _key in _existing:
            continue
        _c.put_object(Bucket=s3_bucket, Key=_key, Body=b'')
        _made += 1
        if _made % 250 == 0:
            print(f'  {_made} 件 ({time.time() - _t0:.0f}s)')

    # 数えなおす(put_object は結果整合ではないが、念のため一覧で確認する)
    _keys = []
    for _p in _paginator.paginate(Bucket=s3_bucket, Prefix=toomany_prefix):
        for _o in _p.get('Contents', []):
            _keys.append(_o['Key'])
    record('S-3-seed.txt',
           f'prefix={toomany_prefix}\n'
           f'created={_made}\n'
           f'total_in_bucket={len(_keys)}\n'
           f'elapsed={time.time() - _t0:.0f}s\n')
    assert len(_keys) == toomany_count, \
        f'{toomany_prefix} の件数が {len(_keys)} で {toomany_count} と違う'
else:
    print('run_toomany=False のためスキップ')

In [ ]:
# S-3 / S-5 用のプロジェクトを作り、S3 アドオン(通常バケット)を接続する
if run_toomany or run_intra:
    async def _step(page):
        await create_project_and_open_files(page, s3_addon_project)
        await connect_s3_addon(page, s3_addon_project, s3_bucket)

    await run_pw(_step)
    dump_wb_responses('S-3-addon-connect.json')
else:
    print('run_toomany / run_intra がどちらも False のためスキップ')

### `toomany` フォルダを開き、底までスクロールして追加読み込みさせる

`handleScroll` はスクロールが最終行に達したときにだけ `fetchData` を呼ぶ。
`#tb-tbody` を底まで送る操作を、読み込み件数が増えなくなるまで繰り返す。

In [ ]:
toomany_pages = []

if run_toomany:
    async def _step(page):
        global toomany_pages
        wb_responses.clear()
        folder = toomany_prefix.rstrip('/')
        await grdm.get_select_folder_title_locator(page, folder).click()
        await asyncio.sleep(3)

        stable = 0
        last = -1
        for i in range(600):
            await page.evaluate(
                "() => { var e = document.querySelector('#tb-tbody');"
                " e.scrollTop = e.scrollHeight; }")
            await asyncio.sleep(0.7)
            loaded = sum(p.get('count', 0) for p in listing_pages(path_contains=folder))
            if loaded == last:
                stable += 1
            else:
                stable = 0
                last = loaded
            if loaded >= toomany_count and stable >= 3:
                break
            if stable >= 15:
                print(f'[!] {loaded} 件で増えなくなった(scroll {i} 回)')
                break
        toomany_pages = listing_pages(path_contains=folder)
        print(f'一覧応答 {len(toomany_pages)} 本、合計 '
              f'{sum(p.get("count", 0) for p in toomany_pages)} 件')

    await run_pw(_step)
else:
    print('run_toomany=False のためスキップ')

In [ ]:
# S-3 の判定
if run_toomany:
    _names = []
    for _p in toomany_pages:
        _names.extend([n for n in _p['names'] if n])
    _dups = sorted({n for n in _names if _names.count(n) > 1}) if len(_names) < 4000 else []
    _missing = sorted(set(toomany_names) - set(_names))
    _extra = sorted(set(_names) - set(toomany_names))
    _last = toomany_pages[-1] if toomany_pages else None

    _verdict = {
        'expected': toomany_count,
        'listed_unique': len(set(_names)),
        'listed_total': len(_names),
        'pages': len(toomany_pages),
        'page_counts': [p.get('count') for p in toomany_pages],
        'token_chain': [
            {'request_next_token': p['request_next_token'],
             'response_next_token': p['response_next_token'],
             'count': p.get('count')}
            for p in toomany_pages],
        'duplicates': _dups,
        'missing': _missing[:50],
        'missing_count': len(_missing),
        'unexpected': _extra[:50],
        'last_response_next_token': _last['response_next_token'] if _last else None,
        'statuses': sorted({p['status'] for p in toomany_pages}),
    }
    record('S-3-listing-pages.json',
           _redact(json.dumps(_verdict, indent=2, ensure_ascii=False)))
    dump_wb_responses('S-3-wb-responses.json')

    assert _verdict['statuses'] and max(_verdict['statuses']) < 400, \
        f'一覧応答にエラーがある: {_verdict["statuses"]}'
    assert not _missing, f'一覧に出てこない名前が {len(_missing)} 件ある: {_missing[:10]}'
    assert not _extra, f'期待しない名前が出ている: {_extra[:10]}'
    assert not _dups, f'一覧に重複がある: {_dups[:10]}'
    assert len(toomany_pages) >= 2, \
        '一覧応答が 1 本しかない。1000 件超のページングが起きていない'
    assert _last['response_next_token'] in (None, ''), \
        f'最終ページに next_token が残っている: {_last["response_next_token"]!r}'
    print(f'S-3: {toomany_count} 件を {len(toomany_pages)} 本の応答で重複・欠落なく取得。'
          '最終ページの next_token は空')
else:
    print('run_toomany=False のためスキップ')

### (任意) 全 1500 行を 1 件ずつ画面でたどる

`取りまとめ-S3共通.ipynb` セル 135 と同じ確認。treebeard は表示範囲の行しか
DOM に置かない(仮想スクロール)ため、**DOM の行数を数えても総数にはならない**。
そこで先頭から順に `scroll_into_view_if_needed` でたどり、全件が画面に出せる
ことを確かめる。1500 回の Playwright 呼び出しになるので数分かかる。
`toomany_walk_rows = False` にすると飛ばせる(判定は上のセルの応答解析で足りる)。

In [ ]:
toomany_walk_rows = True

if run_toomany and toomany_walk_rows:
    async def _step(page):
        _t0 = time.time()
        for i, name in enumerate(toomany_names):
            try:
                await grdm.get_select_file_extension_locator(
                    page, name).scroll_into_view_if_needed(timeout=transition_timeout)
            except Exception as e:
                raise AssertionError(
                    f'{i + 1} 件目 "{name}" を画面に出せない: {e!r}')
            await page.evaluate(
                "() => document.querySelector('#tb-tbody').scrollBy(0, 40)")
            if (i + 1) % 250 == 0:
                print(f'  {i + 1} 件目まで確認 ({time.time() - _t0:.0f}s)')
        _rows = await page.locator('//*[@id = "tb-tbody"]//*[contains(@class, "tb-row")]').count()
        record('S-3-row-walk.txt',
               f'walked={len(toomany_names)}\n'
               f'dom_rows_at_end={_rows}  # 仮想スクロールのため総数ではない\n'
               f'elapsed={time.time() - _t0:.0f}s\n')

    await run_pw(_step)
else:
    print('スキップ(run_toomany=False または toomany_walk_rows=False)')

## S-4: バージョニング有効バケットでの削除(V-1)

バージョニングが有効なバケットでは `DeleteObject` は実体を消さず
**DeleteMarker を積む**。PR #77 の仕様は「UI の削除でその Key の
`Versions` も `DeleteMarkers` も残さない」こと。

手順: 同名ファイルを UI から 3 回アップロード(上書き)→
`list_object_versions` で 3 版以上あることを確認 → UI から削除 →
`list_object_versions` を取り直して **両方 0** であることを確認。

In [ ]:
s4_project = f'{rdm_project_prefix}-versioned'
s4_file_name = 'S-4-versioned.txt'
s4_file_path = os.path.join(work_dir, s4_file_name)


def list_versions(bucket, prefix, region=None):
    """当該 Key の Versions / DeleteMarkers を全ページ集める。"""
    c = s3_client(region or s3_region_default)
    versions, markers = [], []
    kwargs = {'Bucket': bucket, 'Prefix': prefix}
    while True:
        r = c.list_object_versions(**kwargs)
        versions.extend({'Key': v['Key'], 'VersionId': v['VersionId'],
                         'IsLatest': v['IsLatest'], 'Size': v.get('Size')}
                        for v in r.get('Versions', []))
        markers.extend({'Key': m['Key'], 'VersionId': m['VersionId'],
                        'IsLatest': m['IsLatest']}
                       for m in r.get('DeleteMarkers', []))
        if not r.get('IsTruncated'):
            break
        kwargs['KeyMarker'] = r.get('NextKeyMarker')
        kwargs['VersionIdMarker'] = r.get('NextVersionIdMarker')
    return {'prefix': prefix, 'Versions': versions, 'DeleteMarkers': markers,
            'version_count': len(versions), 'delete_marker_count': len(markers)}


if run_versioned:
    async def _step(page):
        await create_project_and_open_files(page, s4_project)
        await connect_s3_addon(page, s4_project, s3_bucket_versioned)

    await run_pw(_step)
    dump_wb_responses('S-4-addon-connect.json')
else:
    print('run_versioned=False のためスキップ')

In [ ]:
if run_versioned:
    async def _step(page):
        await open_storage(page, addon_storage_name)
        for i in range(3):
            with open(s4_file_path, 'w') as f:
                f.write(f'S-4 versioned delete test / revision {i + 1}\n')
                f.write(f'run_id: {run_id}\n')
                f.write(f'at: {datetime.now().isoformat()}\n')
                f.write('x' * (100 * (i + 1)))
            await clear_growls(page)
            await grdm.upload_file(page, s4_file_path)
            await wait_for_upload_finished(page, s4_file_name)
            # 2 回目以降は「上書きしますか」の確認が出る
            keep = page.locator('//button[contains(text(), "置き換える")'
                                ' or contains(text(), "Replace")]')
            if await keep.count() > 0 and await keep.first.is_visible():
                await keep.first.click()
                await wait_for_upload_finished(page, s4_file_name)
            await asyncio.sleep(2)
            print(f'  {i + 1} 回目のアップロード完了')
        await expect(grdm.get_select_file_title_locator(page, s4_file_name)
                     ).to_be_visible(timeout=transition_timeout)

    await run_pw(_step)
else:
    print('run_versioned=False のためスキップ')

In [ ]:
if run_versioned:
    s4_before = list_versions(s3_bucket_versioned, s4_file_name)
    record('S-4-versions-before.json',
           _redact(json.dumps(s4_before, indent=2, ensure_ascii=False)))
    assert s4_before['version_count'] >= 3, (
        f'{s4_file_name} の版が {s4_before["version_count"]} 個しかない。'
        'バージョニングが効いていないか、上書きになっていない')
    print(f'S-4: 削除前 Versions={s4_before["version_count"]} '
          f'DeleteMarkers={s4_before["delete_marker_count"]}')
else:
    print('run_versioned=False のためスキップ')

In [ ]:
# (任意) X9 の実機版: 同一 Key に 1001 版を作ってから削除する
if run_versioned and run_versioned_bulk:
    _c = s3_client(s3_region_default)
    _t0 = time.time()
    _target = 1001
    _have = list_versions(s3_bucket_versioned, s4_file_name)['version_count']
    for i in range(_have, _target):
        _c.put_object(Bucket=s3_bucket_versioned, Key=s4_file_name,
                      Body=f'bulk {i}\n'.encode())
        if (i + 1) % 200 == 0:
            print(f'  {i + 1} 版 ({time.time() - _t0:.0f}s)')
    s4_before = list_versions(s3_bucket_versioned, s4_file_name)
    record('S-4-versions-before.json',
           _redact(json.dumps(s4_before, indent=2, ensure_ascii=False)))
    print(f'S-4 bulk: Versions={s4_before["version_count"]}')
else:
    print('run_versioned_bulk=False のためスキップ(既定)')

In [ ]:
# UI から削除する
if run_versioned:
    async def _step(page):
        await grdm.get_select_file_extension_locator(page, s4_file_name).click()
        delete_btn = page.locator(
            '//i[contains(@class, "fa-trash")]/../*[text() = "削除"]')
        await expect(delete_btn).to_be_enabled(timeout=transition_timeout)
        await clear_growls(page)
        wb_responses.clear()
        await delete_btn.click()
        confirm = page.locator(
            '//*[@id = "tb-tbody"]//*[@class = "modal-content"]'
            '//*[contains(@class, "btn-danger")]')
        await expect(confirm).to_be_enabled(timeout=transition_timeout)
        await asyncio.sleep(1)
        await confirm.click()
        await expect(grdm.get_select_file_title_locator(page, s4_file_name)
                     ).to_have_count(0, timeout=transition_timeout * 3)
        print('growl:', await growl_messages(page))

    await run_pw(_step)
    # 削除は非同期ではないが、S3 の一覧が追いつくまで少し待つ
    time.sleep(5)
else:
    print('run_versioned=False のためスキップ')

In [ ]:
if run_versioned:
    s4_after = list_versions(s3_bucket_versioned, s4_file_name)
    _bad = wb_failures()
    record('S-4-versions-after.json',
           _redact(json.dumps({'after': s4_after,
                               'wb_failures': _bad,
                               'before_version_count': s4_before['version_count'],
                               'before_delete_marker_count':
                                   s4_before['delete_marker_count']},
                              indent=2, ensure_ascii=False)))
    dump_wb_responses('S-4-wb-responses.json')
    assert not _bad, 'S-4 の削除で WB がエラーを返している'
    assert s4_after['version_count'] == 0, (
        f'削除後も Versions が {s4_after["version_count"]} 個残っている。'
        'PR #77 の仕様(全版を消す)を満たしていない')
    assert s4_after['delete_marker_count'] == 0, (
        f'削除後に DeleteMarkers が {s4_after["delete_marker_count"]} 個ある。'
        'DeleteMarker を残す実装になっている')
    print('S-4: Versions / DeleteMarkers ともに 0。V-1 を満たす')
else:
    print('run_versioned=False のためスキップ')

## S-5: intra copy / move(I-2)

同一プロジェクト・同一ストレージ内の移動とコピーは、WB では
`intra_move` / `intra_copy` になり、S3 では `CopyObject`(SigV4 署名)として
出る。ここが壊れると `SignatureDoesNotMatch` か、`x-amz-copy-source` の
署名不一致で失敗する。

fangorn ではドラッグ&ドロップで行う。`getCopyMode`(fangorn.js)が

```
if (folder.data.isPointer || altKey || !canMove) { return 'copy'; }
return 'move';
```

としているので、**素のドラッグ = 移動、Alt を押しながらのドラッグ = コピー**。
POST の本文には `action: "move"` / `"copy"` が入る。

**判定**(指示 §1.7): 成功トースト(「移動に成功しました」「コピーに成功しました」)と、
`wb_responses` の POST が 2xx であること。

ファイルは 6MB。5GB のガード(`can_intra_copy` の `file_size` 判定)は
実機では踏まない。

In [ ]:
# 移動先・コピー先のフォルダを先に作っておく(空キーを置くと一覧でフォルダになる)
if run_intra:
    _c = s3_client(s3_region_default)
    for _p in (intra_copy_dest, intra_move_dest):
        _c.put_object(Bucket=s3_bucket, Key=_p, Body=b'')
    print('作成:', intra_copy_dest, intra_move_dest)

    s5_file_name = 'S-5-intra.bin'
    s5_file_path = os.path.join(work_dir, s5_file_name)
    with open(s5_file_path, 'wb') as f:
        f.write(os.urandom(intra_file_size))
    print(s5_file_path, os.path.getsize(s5_file_path), 'bytes')
else:
    print('run_intra=False のためスキップ')

In [ ]:
# S-3 で 1500 行を読み込んだ状態のツリーは重いので、開き直してから行う
if run_intra:
    async def _step(page):
        await open_project(page, s3_addon_project)
        await open_files_tab(page)
        await open_storage(page, addon_storage_name)
        await clear_growls(page)
        await grdm.upload_file(page, s5_file_path)
        await wait_for_upload_finished(page, s5_file_name)
        await expect(grdm.get_select_file_title_locator(page, s5_file_name)
                     ).to_be_visible(timeout=transition_timeout * 3)
        for _name in (intra_copy_dest.rstrip('/'), intra_move_dest.rstrip('/')):
            await expect(grdm.get_select_folder_title_locator(page, _name)
                         ).to_be_visible(timeout=transition_timeout)

    await run_pw(_step)
    dump_wb_responses('S-5-upload.json')
else:
    print('run_intra=False のためスキップ')

In [ ]:
async def drag_and_drop_with_alt(page, source, dest):
    """Alt を押しながらドラッグする(= fangorn の copyMode を 'copy' にする)。

    fangorn.js は `$(document).keydown` で `e.altKey` を見て module 変数
    `altKey` を立てる。Playwright の keyboard.down は実キーイベントを出すので
    このハンドラが反応する。mouse.up のあとに離すこと(離すのが早いと
    `getCopyMode` が呼ばれる時点で false に戻っている)。
    """
    await page.keyboard.down('Alt')
    try:
        await grdm.drag_and_drop(page, source, dest)
    finally:
        await page.keyboard.up('Alt')


s5_results = {}

if run_intra:
    async def _step(page):
        # --- コピー: Alt + ドラッグ
        await clear_growls(page)
        wb_responses.clear()
        src = grdm.get_select_file_draggable_locator(page, s5_file_name)
        dst = grdm.get_select_folder_droppable_locator(page, intra_copy_dest.rstrip('/'))
        await drag_and_drop_with_alt(page, src, dst)
        await asyncio.sleep(5)
        s5_results['copy'] = {
            'growls': await growl_messages(page),
            'wb': [e for e in wb_responses if e['method'] == 'POST'],
        }
        # コピーなので元のファイルは残っている
        await expect(grdm.get_select_file_title_locator(page, s5_file_name)
                     ).to_be_visible(timeout=transition_timeout)

    await run_pw(_step)
    print(s5_results['copy']['growls'])
else:
    print('run_intra=False のためスキップ')

In [ ]:
if run_intra:
    async def _step(page):
        # --- 移動: 素のドラッグ
        await clear_growls(page)
        wb_responses.clear()
        src = grdm.get_select_file_draggable_locator(page, s5_file_name)
        dst = grdm.get_select_folder_droppable_locator(page, intra_move_dest.rstrip('/'))
        await grdm.drag_and_drop(page, src, dst)
        await asyncio.sleep(5)
        s5_results['move'] = {
            'growls': await growl_messages(page),
            'wb': [e for e in wb_responses if e['method'] == 'POST'],
        }

    await run_pw(_step)
    print(s5_results['move']['growls'])
else:
    print('run_intra=False のためスキップ')

In [ ]:
# S-5 の判定: S3 側の実体と、POST の action / ステータス
if run_intra:
    _c = s3_client(s3_region_default)

    def _head(key):
        try:
            r = _c.head_object(Bucket=s3_bucket, Key=key)
            return {'exists': True, 'ContentLength': r['ContentLength']}
        except Exception as e:
            return {'exists': False, 'error': type(e).__name__}

    _s3_state = {
        'root': _head(s5_file_name),
        'copy_dest': _head(intra_copy_dest + s5_file_name),
        'move_dest': _head(intra_move_dest + s5_file_name),
    }

    def _actions(entries):
        out = []
        for e in entries:
            body = e.get('body') or ''
            out.append({'status': e['status'], 'url': e['url'],
                        'has_copy': '"copy"' in body or "'copy'" in body,
                        'has_move': '"move"' in body or "'move'" in body})
        return out

    _verdict = {
        's3': _s3_state,
        'copy': {'growls': s5_results.get('copy', {}).get('growls'),
                 'posts': _actions(s5_results.get('copy', {}).get('wb', []))},
        'move': {'growls': s5_results.get('move', {}).get('growls'),
                 'posts': _actions(s5_results.get('move', {}).get('wb', []))},
        'expected_file_size': intra_file_size,
    }
    record('S-5-wb-responses.json',
           _redact(json.dumps(_verdict, indent=2, ensure_ascii=False)))
    dump_wb_responses('S-5-wb-responses-raw.json')

    for _op, _word in (('copy', 'コピー'), ('move', '移動')):
        _posts = _verdict[_op]['posts']
        assert _posts, f'{_op} の POST が観測できていない(ドラッグが届いていない)'
        _bad = [p for p in _posts if p['status'] >= 400]
        assert not _bad, f'{_op} の POST が失敗している: {_bad}'
        _growls = _verdict[_op]['growls'] or []
        assert any(f'{_word}に成功しました' in g for g in _growls), \
            f'{_op} の成功トースト(「{_word}に成功しました」)が出ていない: {_growls}'

    assert _s3_state['copy_dest']['exists'], 'コピー先に実体が無い'
    assert _s3_state['move_dest']['exists'], '移動先に実体が無い'
    assert not _s3_state['root']['exists'], '移動したのに元の場所にファイルが残っている'
    assert _s3_state['copy_dest']['ContentLength'] == intra_file_size, \
        'コピー先のサイズが違う(CopyObject が壊れている)'
    assert _s3_state['move_dest']['ContentLength'] == intra_file_size, \
        '移動先のサイズが違う'
    print('S-5: copy / move ともに成功し、S3 の実体も期待どおり')
else:
    print('run_intra=False のためスキップ')

## 後処理

1. 証跡の INDEX を作る
2. Playwright を閉じる(ここで HAR・動画・console.log が書き出される)
3. HAR を読んで S-1(G-10)の 302 → presigned URL を判定する
4. バケットの残骸を消す
5. 後始末チェックリストを消化する(機関ストレージを戻すのは**手動**)

In [ ]:
# 証跡の INDEX
_files = sorted(os.listdir(run_dir))
_index = ['# 証跡 INDEX',
          '',
          f'- run_id: {run_id}',
          f'- 取得日時: {datetime.now().isoformat()}',
          f'- 対象: RCOSDP/RDM-waterbutler#93 ({wb_deployed_sha}) '
          f'/ RCOSDP/RDM-osf.io#746 ({osf_deployed_sha})',
          '',
          '| ファイル | サイズ | 対応シナリオ |',
          '|---|---|---|']
_scenario = {'S-0': '環境確認 (IF-5/R-5)', 'S-1': '機関ストレージ基本操作 (IF-3既定/G-10)',
             'S-2': '明示リージョン (IF-3)', 'S-3': '1000件超一覧 (P-6/R-6)',
             'S-4': 'バージョン削除 (V-1)', 'S-5': 'intra copy/move (I-2)',
             'S-6': 'CompleteMultipartUpload Error Code (E-1)'}
for _f in _files:
    _key = _f[:3]
    _index.append(f'| `{_f}` | {os.path.getsize(os.path.join(run_dir, _f))} | '
                  f'{_scenario.get(_key, "Playwright 成果物")} |')
_index.append('')
_index.append('`har.zip` / `video-*.webm` / `console.log` / `last-*.{png,html}` は '
              '`scripts/playwright.py` が `finish_pw_context` で書き出したもの。')
record('INDEX.md', '\n'.join(_index))

In [ ]:
# Playwright を閉じる。**これをしないと HAR が書き出されない。**
await finish_pw_context(timeout=300)

### S-1 (G-10): HAR からダウンロードのリダイレクトを判定する

WB の `download_file` は `accept_url` が真のとき presigned URL を文字列で返し、
ハンドラが `self.redirect(stream)` する
(`waterbutler/server/api/v1/provider/metadata.py`)。

**判定**: ダウンロード要求に対する応答が **302** で、`Location` が
`amazonaws.com` かつ `X-Amz-Signature` を含むこと(= SigV4 の presigned URL)。

パスの `providers/` の後ろは、機関ストレージ経由なら `osfstorage`、
アドオン経由なら `s3` になる(セル [5] の表のとおり、`osfstorage` は
`download` だけは内側の `s3` に委譲するので、どちらでも 302 になる)。

In [ ]:
import re
import zipfile

_har_zip = os.path.join(run_dir, 'har.zip')
har_verdict = {'har_zip': _har_zip, 'exists': os.path.exists(_har_zip)}

if har_verdict['exists']:
    with zipfile.ZipFile(_har_zip) as _z:
        _member = [n for n in _z.namelist() if n.endswith('.har')][0]
        _har = json.loads(_z.read(_member).decode('utf-8'))

    _redirects = []
    for _e in _har['log']['entries']:
        _req, _res = _e['request'], _e['response']
        if _req['method'] != 'GET':
            continue
        if '/v1/resources/' not in _req['url']:
            continue
        if not re.search(r'/providers/(osfstorage|s3)/', _req['url']):
            continue
        if 'meta=' in _req['url'] or 'zip=' in _req['url']:
            continue
        _loc = ''
        for _h in _res.get('headers', []):
            if _h['name'].lower() == 'location':
                _loc = _h['value']
        _redirects.append({
            'url': _req['url'],
            'status': _res['status'],
            'location_host': _loc.split('/')[2] if '://' in _loc else '',
            'location_is_amazonaws': 'amazonaws.com' in _loc,
            'location_has_signature': 'X-Amz-Signature' in _loc,
            'location_has_sigv4_algorithm': 'AWS4-HMAC-SHA256' in _loc,
            'location': re.sub(r'(X-Amz-(?:Signature|Credential|Security-Token)=)'
                               r'[^&\s]+', r'\1***', _loc),
        })

    _ok = [r for r in _redirects
           if r['status'] == 302 and r['location_is_amazonaws']
           and r['location_has_signature']]
    har_verdict['entries'] = len(_har['log']['entries'])
    har_verdict['download_candidates'] = _redirects
    har_verdict['presigned_redirects'] = len(_ok)
    record('S-1-download-redirect.json',
           _redact(json.dumps(har_verdict, indent=2, ensure_ascii=False)))
    assert _ok, (
        'HAR に「302 かつ Location が X-Amz-Signature 付きの amazonaws.com」'
        'である応答が無い。決定-19(accept_url で presigned URL に飛ばす)が'
        '効いていない可能性がある。S-1-download-redirect.json を見ること')
    print(f'S-1 (G-10): presigned URL への 302 を {len(_ok)} 件確認')
else:
    record('S-1-download-redirect.json',
           json.dumps(har_verdict, indent=2, ensure_ascii=False))
    raise AssertionError(
        f'{_har_zip} が無い。default_result_path が run_dir を指していないか、'
        'finish_pw_context が走っていない')

In [ ]:
# バケットの残骸を消す(toomany / S-4 / S-5)
_cleanup = []


def _delete_prefix(bucket, prefix, region):
    c = s3_client(region)
    deleted = 0
    paginator = c.get_paginator('list_objects_v2')
    batch = []
    for p in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for o in p.get('Contents', []):
            batch.append({'Key': o['Key']})
            if len(batch) == 1000:
                c.delete_objects(Bucket=bucket, Delete={'Objects': batch})
                deleted += len(batch)
                batch = []
    if batch:
        c.delete_objects(Bucket=bucket, Delete={'Objects': batch})
        deleted += len(batch)
    return deleted


def _delete_all_versions(bucket, prefix, region):
    """バージョニング有効バケット用。Versions と DeleteMarkers の両方を消す。"""
    c = s3_client(region)
    deleted = 0
    while True:
        r = c.list_object_versions(Bucket=bucket, Prefix=prefix)
        objs = [{'Key': v['Key'], 'VersionId': v['VersionId']}
                for v in r.get('Versions', []) + r.get('DeleteMarkers', [])]
        if not objs:
            break
        for i in range(0, len(objs), 1000):
            c.delete_objects(Bucket=bucket, Delete={'Objects': objs[i:i + 1000]})
        deleted += len(objs)
    return deleted


if run_toomany:
    _n = _delete_prefix(s3_bucket, toomany_prefix, s3_region_default)
    _cleanup.append(f'{toomany_prefix}: {_n} 件削除')
if run_intra:
    for _p in (intra_copy_dest, intra_move_dest):
        _n = _delete_prefix(s3_bucket, _p, s3_region_default)
        _cleanup.append(f'{_p}: {_n} 件削除')
    _n = _delete_prefix(s3_bucket, s5_file_name, s3_region_default)
    _cleanup.append(f'{s5_file_name}: {_n} 件削除')
if run_versioned:
    _n = _delete_all_versions(s3_bucket_versioned, s4_file_name, s3_region_default)
    _cleanup.append(f'{s4_file_name} (全版): {_n} 件削除')

_left = s3_client(s3_region_default).list_objects_v2(Bucket=s3_bucket, MaxKeys=20)
_cleanup.append(f's3_bucket の残り: KeyCount={_left.get("KeyCount", 0)} '
                f'{[o["Key"] for o in _left.get("Contents", [])]}')
record('S-3-cleanup.txt', '\n'.join(_cleanup))

### 後始末チェックリスト

**この証跡取りは機関ストレージの設定を 2 回上書きしている。**
`S-0-institutional-storage-before.txt` に登録前の状態が残っているので、
それを見ながら次を消化すること。

- [ ] 機関ストレージを元の設定に戻す(**手動**。管理画面で元のプロバイダを
      選び直して保存する。「登録を外す」操作は無い)
- [ ] 本ノートブックが作ったプロジェクトを削除する
      (`*-institutional` / `*-apne1` / `*-addon` / `*-versioned`)
- [ ] ユーザー設定 →「アドオンアカウント構成」から S3 アカウントの連携を解除する
      (アドオンのアカウントは**ユーザー単位**で残る)
- [ ] バケットが空になっていることを確認する(上のセルの `S-3-cleanup.txt`)
- [ ] `S-0-jenkins-build.txt` を手で保存したか確認する
- [ ] 証跡ディレクトリを `aws-s3-sigv4-improvement/evidence/e2e/` 配下に置いたか確認する
- [ ] S-6(`scripts/s3_complete_error_codes.py`)を実行し、
      `S-6-complete-error-codes.json` を同じ証跡ディレクトリに入れる

In [ ]:
# 残っているプロジェクトの一覧を出す(削除は手動で行う)
print('削除対象のプロジェクト:')
for _p in [f'{rdm_project_prefix}-institutional', f'{rdm_project_prefix}-apne1',
           s3_addon_project, s4_project]:
    print(' -', _p)
print()
print('証跡:', run_dir)
print('作業ディレクトリを削除します:', work_dir)

In [ ]:
!rm -fr {work_dir}